# Document Scanner — Colab GPU Corner Detection Training Launcher

Runs the corner detection comparison between Approach A and Approach B (`[REQ-30]`, `[REQ-31]`, ADR-007) on a Colab GPU (T4):

| Run | Model | Formulation | Loss | Target |
|---|---|---|---|---|
| exp-009 | `CornerRegNet` | Approach A: Direct Coordinate Regression | L1 | 8 normalized coordinates in [0, 1] |
| exp-010 | `CornerHeatmapNet` | Approach B: Heatmap Regression | MSE | 4-channel 512x512 Gaussians (sigma=8) |

### Fairness Commitments (ADR-007)
1. **Shared Encoder Backbone**: Both models use the identical 4-level U-Net encoder (`base_channels=64, levels=4`).
2. **No Global Average Pooling for Approach A**: `CornerRegNet` reduces bottleneck features to an 8x8 spatial grid before FC layers, preserving 2D spatial layout.
3. **Single Shared Data Stream**: Both arms step on the exact same synthetic batches in the same order.
4. **Equal LR & Budget**: Equal learning rate search effort (`1e-3` Adam, Cosine Annealing, 40 epochs).
5. **Zero Regularization**: `dropout=0.0` and `weight_decay=0.0` (`[CON-04]`).

### Step 0: Confirm CUDA GPU Availability

Ensures a CUDA GPU (e.g. T4) is active on Colab.

In [ ]:
import os, torch
print('PyTorch version:', torch.__version__)
print('vCPUs:', os.cpu_count())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM: %.1f GB' % (torch.cuda.get_device_properties(0).total_memory / 1e9))
else:
    raise SystemExit('No CUDA device. Select Runtime -> Change runtime type -> T4 GPU, then restart.')

### Step 1: Mount Google Drive & Clone Repository

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
if not os.path.exists('/content/DocEn'):
    !git clone https://github.com/HedieTahmouresi/DocEn.git /content/DocEn
%cd /content/DocEn
!git pull
!git log --oneline -1

### Step 2: Install Dependencies & Extract Data

In [ ]:
!pip install -q -r requirements.txt

import os
if not os.path.exists('data/clean_scans'):
    for src in ('/content/drive/MyDrive/data.zip', '/content/data.zip'):
        if os.path.exists(src):
            print('Extracting data from:', src)
            !unzip -q "{src}" -d .
            break
    else:
        raise SystemExit('data.zip not found in Drive or /content.')

if not os.path.exists('data/frozen/val'):
    print('Frozen evaluation sets missing - generating them now...')
    !python -m src.data.freeze

!ls data && ls data/frozen

### Step 3: Run Unit Tests & Verification

Verifies that `CornerRegNet`, `CornerHeatmapNet`, heatmap rendering, soft-argmax subpixel extraction, and corner metrics pass cleanly.

In [ ]:
!python -m pytest tests/test_corner_pipeline.py -v

### Step 3b: Smoke Run — 2 Epochs, 200 Samples

Validates training loop, loaders, checkpointing, and metric logging before committing to full 40-epoch training.

In [ ]:
!python train_corners.py --env colab_t4 --epochs 2 --samples-per-epoch 200
!head -5 runs/exp-009_corner_approach_a/metrics.csv
!head -5 runs/exp-010_corner_approach_b/metrics.csv
!rm -rf runs/exp-009* runs/exp-010*

### Step 4: Checkpoint Durability & Google Drive Mirroring

Configures periodic syncing of checkpoints (`runs/`) to Google Drive every 5 epochs.

In [ ]:
import os, shutil

DRIVE_RUNS = '/content/drive/MyDrive/DocEn_runs'
os.makedirs(DRIVE_RUNS, exist_ok=True)
print('Drive mirror path:', DRIVE_RUNS)
print('Existing runs in Drive:', sorted(os.listdir(DRIVE_RUNS)))

RESUMING = False
if RESUMING:
    for name in os.listdir(DRIVE_RUNS):
        if 'corner' in name or 'exp-009' in name or 'exp-010' in name:
            shutil.copytree(os.path.join(DRIVE_RUNS, name), os.path.join('runs', name), dirs_exist_ok=True)
    print('Restored runs:', sorted(os.listdir('runs')))

### Step 5: Execute Phase 06 Corner Detection Training

Trains both Approach A and Approach B side-by-side over 40 epochs.

In [ ]:
!python train_corners.py --env colab_t4 \
    --mirror-dir /content/drive/MyDrive/DocEn_runs --mirror-every 5

### Step 6: Evaluate & Compare Corner Models

Evaluates trained checkpoints on synthetic test set and real smartphone photos, producing summary reports and comparison figures.

In [ ]:
!python -m scripts.evaluate_corners

### Step 7: Package & Download Results

In [ ]:
!zip -r phase06_corner_results.zip outputs/figures/ p06_* runs/exp-009*/metrics.csv runs/exp-010*/metrics.csv
!cp phase06_corner_results.zip /content/drive/MyDrive/

from google.colab import files
files.download('phase06_corner_results.zip')